## Imports

In [1]:
import sys
sys.path.insert(0, "../..")
from data.dataprocessor import DataProcessor
from data.loader import DataLoader
from models.embedder import BERTEmbedder
from models.datasets import TextDataset
from utilities.utils import generate_interactive_3d_plot
from recommender.recommender import Recommender

import plotly.express as px
from sklearn.decomposition import PCA

from sys import path
import os

import numpy as np

import torch
from torch.utils.tensorboard import SummaryWriter

## Data Loading and Preprocessing

In [2]:
path_str = "D:\\Projects\\Recommender\\Dataset\\manhwa_mal.csv\\manhwa_mal.csv"
data = DataLoader.load_from_csv(path_str)
data.head()

2026-07-16 02:06:22,722 - INFO - Ingesting CSV data from D:\Projects\Recommender\Dataset\manhwa_mal.csv\manhwa_mal.csv
2026-07-16 02:06:22,756 - INFO - Successfully loaded 2943 rows.


,Unnamed: 0,type,title,chapters,status,genres,favorites,popularity,rank,score,members,synopsis,volumns,authors,publish_time
0,0,manhwa,Solo Leveling,201,Finished,"Action,Adventure,Fantasy","40,014",#7,#56,8.68,"431,289","Ten years ago, ""the Gate"" appeared and connect...",Unknown,"Chugong (Story), Jang, Sung-rak (Art), Discipl...","Mar 4, 2018 to May 31, 2023"
1,1,manhwa,The Horizon,21,Finished,"Adventure,Drama","4,047",#187,#58,8.67,"75,806","In a world ravaged by war, a young boy walks d...",3,"Jeong, Ji-Hoon (Story & Art)","Mar 30, 2016 to Jul 21, 2016"
2,2,manhwa,Wind Breaker,Unknown,Publishing,"Action,Drama,Sports","2,688",#368,#94,8.58,"42,434","Burdened with expectations since childhood, se...",Unknown,"Jo, Yongseok (Story & Art)","Dec 15, 2013 to ?"
3,3,manhwa,Bastard,94,Finished,"Drama,Horror,Mystery,Romance","6,455",#84,#140,8.50,"126,088",There is nowhere that Seon Jin can find solace...,5,"Kim, Carnby (Story), Hwang, Young-chan (Art)","Jul 4, 2014 to May 6, 2016"
4,4,manhwa,Who Made Me a Princess,125,Finished,"Comedy,Fantasy,Romance","2,648",#349,#175,8.44,"44,428","In the novel The Lovely Princess, the secondar...",9,"Plutus (Story), Spoon (Art)","Dec 20, 2017 to Apr 30, 2022"


In [3]:
processor = DataProcessor()
data_cleaned =  processor.process(data)
data_cleaned.head()

2026-07-16 02:06:22,783 - INFO - Filled NaNs with 0 in column: score
2026-07-16 02:06:22,786 - INFO - Schema validation, imputation, and format checking passed.
2026-07-16 02:06:22,799 - INFO - Processing complete. Final dataset shape: (2683, 15)


,unnamed: 0,type,title,chapters,status,genres,favorites,popularity,rank,score,members,synopsis,volumns,authors,publish_time
0,0,manhwa,Solo Leveling,201,Finished,"Action,Adventure,Fantasy","40,014",#7,#56,8.68,"431,289","Ten years ago, ""the Gate"" appeared and connect...",Unknown,"Chugong (Story), Jang, Sung-rak (Art), Discipl...","Mar 4, 2018 to May 31, 2023"
1,1,manhwa,The Horizon,21,Finished,"Adventure,Drama","4,047",#187,#58,8.67,"75,806","In a world ravaged by war, a young boy walks d...",3,"Jeong, Ji-Hoon (Story & Art)","Mar 30, 2016 to Jul 21, 2016"
2,2,manhwa,Wind Breaker,Unknown,Publishing,"Action,Drama,Sports","2,688",#368,#94,8.58,"42,434","Burdened with expectations since childhood, se...",Unknown,"Jo, Yongseok (Story & Art)","Dec 15, 2013 to ?"
3,3,manhwa,Bastard,94,Finished,"Drama,Horror,Mystery,Romance","6,455",#84,#140,8.50,"126,088",There is nowhere that Seon Jin can find solace...,5,"Kim, Carnby (Story), Hwang, Young-chan (Art)","Jul 4, 2014 to May 6, 2016"
4,4,manhwa,Who Made Me a Princess,125,Finished,"Comedy,Fantasy,Romance","2,648",#349,#175,8.44,"44,428","In the novel The Lovely Princess, the secondar...",9,"Plutus (Story), Spoon (Art)","Dec 20, 2017 to Apr 30, 2022"


In [4]:
(data.shape, data_cleaned.shape)

((2943, 15), (2683, 15))

## Text Embeddings

### 1. BERT Base Uncased Embeddings

In [5]:
bert_embedder = BERTEmbedder()
bert_embedder

c:\Users\krato\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [6]:
embeddings = bert_embedder.generate_embeddings(data_cleaned['synopsis'])
embeddings.shape

Extracting: 100%|██████████| 42/42 [00:37<00:00,  1.11it/s]


(2683, 768)

In [7]:
data_cleaned['embedding'] = list(embeddings)
data_cleaned.head()

,unnamed: 0,type,title,chapters,status,genres,favorites,popularity,rank,score,members,synopsis,volumns,authors,publish_time,embedding
0,0,manhwa,Solo Leveling,201,Finished,"Action,Adventure,Fantasy","40,014",#7,#56,8.68,"431,289","Ten years ago, ""the Gate"" appeared and connect...",Unknown,"Chugong (Story), Jang, Sung-rak (Art), Discipl...","Mar 4, 2018 to May 31, 2023","[-0.25960582, 0.025964124, 0.29473713, -0.1175..."
1,1,manhwa,The Horizon,21,Finished,"Adventure,Drama","4,047",#187,#58,8.67,"75,806","In a world ravaged by war, a young boy walks d...",3,"Jeong, Ji-Hoon (Story & Art)","Mar 30, 2016 to Jul 21, 2016","[-0.017733792, -0.096118085, 0.24612711, -0.13..."
2,2,manhwa,Wind Breaker,Unknown,Publishing,"Action,Drama,Sports","2,688",#368,#94,8.58,"42,434","Burdened with expectations since childhood, se...",Unknown,"Jo, Yongseok (Story & Art)","Dec 15, 2013 to ?","[-0.28088892, -0.009417185, 0.29342905, -0.224..."
3,3,manhwa,Bastard,94,Finished,"Drama,Horror,Mystery,Romance","6,455",#84,#140,8.50,"126,088",There is nowhere that Seon Jin can find solace...,5,"Kim, Carnby (Story), Hwang, Young-chan (Art)","Jul 4, 2014 to May 6, 2016","[-0.26607847, -0.027077071, 0.14012776, -0.219..."
4,4,manhwa,Who Made Me a Princess,125,Finished,"Comedy,Fantasy,Romance","2,648",#349,#175,8.44,"44,428","In the novel The Lovely Princess, the secondar...",9,"Plutus (Story), Spoon (Art)","Dec 20, 2017 to Apr 30, 2022","[-0.31436145, -0.09806787, 0.25833765, -0.1454..."


### 2. Visualizing BERT embeddings

In [3]:
generate_interactive_3d_plot(artifacts_dir="artifacts", output_file="index.html")

2026-07-16 02:27:18,773 - INFO - Loading artifacts...
2026-07-16 02:27:18,853 - INFO - Running UMAP dimensionality reduction...
c:\Users\krato\AppData\Local\Programs\Python\Python310\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
2026-07-16 02:27:37,803 - INFO - Generating custom HTML and JavaScript payload...
2026-07-16 02:27:37,813 - INFO - Success! Interactive multi-select visualization saved to index.html


In [ ]:
recommender

### Saving Embeddings and cleaned data

In [ ]:
# data_cleaned.head()

NameError: name 'data_cleaned' is not defined

In [3]:
artifact_path = os.path.join(os.path.abspath("."), "artifacts" ) # Get the current working directory to confirm where the artifacts will be saved
os.makedirs(artifact_path, exist_ok=True) # Create the artifacts directory if it doesn't exist



clean_data_path = os.path.join(artifact_path, "clean_manhwa_data.parquet")
# data_cleaned.to_parquet(clean_data_path, index=False)

# Save the heavy numpy matrix efficiently
embeddings_path = os.path.join(artifact_path, "synopsis_embeddings.npy")
# np.save(embeddings_path, embeddings)

### Recommender

In [7]:
reco = Recommender()

2026-07-16 05:23:47,934 - INFO - Successfully loaded 2683 records. Embedding matrix shape: (2683, 768)


In [10]:
res = reco.recommend("The Horizon", cosine_weight=0.82, jaccard_weight=0.18)
res

{'query': 'The Horizon',
 'query_genres': 'Adventure,Drama',
 'weights': {'cosine': 0.82, 'jaccard': 0.18},
 'results_count': 5,
 'recommendations': [{'title': 'For the Sake of Sita',
   'genres': 'Drama,Romance',
   'hybrid_score': 0.8114,
   'cosine_score': 0.9163,
   'jaccard_score': 0.3333,
   'synopsis': "For her sake, he would sacrifice anything...\n\r\nWhile volunteering in Nepal, a Korean medical student named Sangmin Han encounters the beautiful yet tragic Sita. Previously, she was a Kumari—a prepubescent girl worshipped as the vessel of a goddess. But she has since been forsaken as a Kumari, and cast out from society. Falling in love, they marry and move to Korea, but their blissful love is short-lived; Sita dies only a year later.\n\r\nLosing the woman dearest to him, Sangmin desperately begs the gods to bring Sita back. His prayers are answered but in exchange for a sacrifice, and he awakens in the streets of Nepal. However, something is amiss—his body has suddenly aged, an

In [11]:
reco.df[reco.df['title'] == "The Horizon"].iloc[0]

unnamed: 0                                                      1
type                                                       manhwa
title                                                 The Horizon
chapters                                                       21
status                                                   Finished
genres                                            Adventure,Drama
favorites                                                   4,047
popularity                                                   #187
rank                                                          #58
score                                                        8.67
members                                                    75,806
synopsis        In a world ravaged by war, a young boy walks d...
volumns                                                         3
authors                              Jeong, Ji-Hoon (Story & Art)
publish_time                       Mar  30, 2016 to Jul  21, 2016
embedding 

In [12]:
res['recommendations'][2]

{'title': 'At the End of the Road',
 'genres': 'Love,Drama,Suspense',
 'hybrid_score': 0.7935,
 'cosine_score': 0.9128,
 'jaccard_score': 0.25,
 'synopsis': "All alone, 18-year-old Yoon Taemin has had to fend for himself his whole life and learn how to survive in a cruel world. He tries to shake off an unusually vivid dream of a boy committing suicide and getting hit by a speeding truck. But the dream turns out to be a vision of his fate—as he dies the exact same way.\n\r\nFollowing the accident, Taemin wakes up in the body of Min Siwoon, a meek and frail teenager who jumped off a building. Although Siwoon is surrounded by wealth and loving parents, it is soon evident that everyone, even his brother, absolutely despises and viciously bullies him since he is unable to stand up for himself.\n\r\nAccepting his new situation, Taemin lives on as Siwoon, protecting himself under the excuse that he lost all his memories. However, his past life quickly catches up with him when he encounters Ha

In [27]:
reco.df[reco.df['title'] == res['recommendations'][2]['title']]['synopsis'].item()

"For her sake, he would sacrifice anything...\n\r\nWhile volunteering in Nepal, a Korean medical student named Sangmin Han encounters the beautiful yet tragic Sita. Previously, she was a Kumari—a prepubescent girl worshipped as the vessel of a goddess. But she has since been forsaken as a Kumari, and cast out from society. Falling in love, they marry and move to Korea, but their blissful love is short-lived; Sita dies only a year later.\n\r\nLosing the woman dearest to him, Sangmin desperately begs the gods to bring Sita back. His prayers are answered but in exchange for a sacrifice, and he awakens in the streets of Nepal. However, something is amiss—his body has suddenly aged, and standing in front of him is a young Sita, with no memory of him.\n\r\nThrough two lovers bound together by a destiny running deeper than it first appears, For the Sake of Sita is the story of Sita growing up as a Kumari, and Sangmin's tribulations in trying to avert her terrible fate.\n\r\n[Written by MAL Re